<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c34-header.png" alt="Nextia Learning · Anomaly Detection: Find What Is Unusual" width="100%">

# Features decide more than the algorithm

**[Features decide more than the algorithm](https://learning.nextia-ai.com/courses/anomaly-detection/m04/features-decide-more-than-the-algorithm/)** · Anomaly Detection: Find What Is Unusual · Module 4, lesson 5 of 5 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** build behaviour features that compare a payment with the customer's own past, rerun the four detectors of this module on raw and on behaviour features, and measure what gave the larger gain: the algorithm or the features.

| | |
|---|---|
| **Time** | About 40 minutes |
| **Needs** | An internet connection for the setup cells (about 7 MB of data, and PyOD with its helper numba). No account. The notebook takes about two minutes to run. |
| **You should know** | The four detectors of this module, from [Far from the neighbours: kNN distance](https://learning.nextia-ai.com/courses/anomaly-detection/m04/far-from-the-neighbours-knn-distance/) to [One-Class SVM and the PyOD toolbox](https://learning.nextia-ai.com/courses/anomaly-detection/m04/one-class-svm-and-the-pyod-toolbox/). |
| **You do not need** | A local project. The setup below downloads everything. |
| **Tested** | Python 3.12.13 in Jupyter on macOS (Apple silicon), from a fresh start (*Restart and run all*), with the package versions of the setup cell |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/anomaly-detection/m04/features-decide-more-than-the-algorithm/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/notebooks/M04-L05-features-decide-more-than-the-algorithm/features-decide-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, numpy 2.5.4, scipy 1.18.1, scikit-learn 1.9.1, matplotlib 3.11.2, pyod 3.6.7. It needs an internet connection. The first time, it takes up to a minute.

You should see one line per package, with the same versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. A restart is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 pyod==3.6.7
from importlib.metadata import version
for p in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib', 'pyod']:
    print(p, version(p))

## Setup: get the course data

Run the next cell. It downloads the data that this lesson uses (`payments.csv.gz`, `sealed-key.txt`) into a folder `anomaly-course/data`, and checks each file's checksum. It also downloads `compare.py` and `features.py`, the course's shared code. It needs an internet connection and no account. It takes about 10 seconds.

You should see `Ready:` and the files with their sizes. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `anomaly-course` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course: no real people, cards or payments. [The dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/data/dataset-card.md) describes every table.

In [ ]:
# Setup: download the course data into anomaly-course/data/ and check each file.
# Standard library only.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C34/"
FILES = {
    "payments.csv.gz": "c4e18c1bbaf422cfc99821f2a1b65cf0e8274f835c40ca757a37a531eeccd542",
    "sealed-key.txt": "144cfd16737bdfcc4e992ad8ede0b8856265a2193c89fa7266fe0b27cb62ff98",
}
CODE_FILES = ["compare.py", "features.py"]
HERE = Path.cwd()
# The notebook works in its own folder anomaly-course/, marked with a file, so it never writes into another project.
if Path(".nextia-notebook").exists():          # the cell ran before: start again from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("anomaly-course").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, else download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder anomaly-course and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, WORK / "data" / name, expected)
for name in CODE_FILES:
    fetch(name, LABS + name, WORK / name)
os.chdir(WORK)
print("Ready:", ", ".join(f"{n} ({(Path('data') / n).stat().st_size / 1e6:.1f} MB)" for n in FILES))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd`, NumPy as `np` and matplotlib as `plt`. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it, and the chart style.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.figsize": (9, 3.2), "axes.spines.top": False, "axes.spines.right": False})

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple, set)):
            return [norm(x) for x in (sorted(v, key=str) if isinstance(v, set) else v)]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. One customer, by hand

A bank clerk who knows a customer notices at once when something is off: "She never pays at night, and never from abroad." The raw payment does not say that. Only the payment **compared with the customer's past** does.

Run the next cell. It loads the payments and shows the last 11 payments of customer C-01234: three ordinary Saturday baskets, then eight payments in Portugal.

In [ ]:
import compare, features
payments = pd.read_csv("data/payments.csv.gz", parse_dates=["ts"])
reference, evaluation = compare.split(payments)
one = payments[payments.customer_id == "C-01234"].sort_values("ts").reset_index(drop=True)
print(one[["ts", "amount", "country", "device_id", "hour"]].tail(11).to_string())

Run the next cell. It builds the behaviour features for this customer by hand, one step at a time. Every step uses only the payments **before** (or at) the payment's own time, as a live detector would.

> **Check.** In the last 8 rows, you should see `new_country` 1 and `new_device` 1, `night` 1, `payments_last_hour` going from 1 to 8, and `minutes_since_last` near 1.

In [ ]:
f = pd.DataFrame({"ts": one.ts, "amount": one.amount})
# 1. the amount against the customer's usual amount: the median of the last 10 payments BEFORE this one
usual = one.amount.shift().rolling(10, min_periods=1).median()
f["usual"] = usual
f["log_amount_vs_usual"] = np.log(one.amount / usual.fillna(one.amount))
# 2. a country (or device) that the customer never used before today (not on the customer's first day)
day = one.ts.dt.floor("D")
first_day = day.min()
for col, name in [("country", "new_country"), ("device_id", "new_device")]:
    first_seen = one.groupby(col).ts.transform("min").dt.floor("D")
    f[name] = ((first_seen == day) & (first_day < day)).astype(int)
# 3. time since the last payment, and payments in the last hour (this one included)
f["minutes_since_last"] = one.ts.diff().dt.total_seconds() / 60
f["payments_last_hour"] = one.set_index("ts").amount.rolling("3600s").count().to_numpy()
f["night"] = one.hour.between(0, 5).astype(int)
print(f.tail(11).to_string(float_format=lambda v: f"{v:.3f}"))

Now compare with the course's `features.behaviour()`, which does the same for all payments at once. Run the next cell. It takes about 10 seconds.

> **Check.** You should see `same as features.py: True True True`.

In [ ]:
behaviour = features.behaviour(payments)          # the same features for all 343,074 payments (about 10 s)
mine = behaviour.loc[payments.customer_id == "C-01234"].reset_index(drop=True)
print("same as features.py:", np.allclose(mine.log_amount_vs_usual, f.log_amount_vs_usual),
      (mine.new_country == f.new_country).all(), (mine.payments_last_hour == f.payments_last_hour).all())

## 2. Eight rows: four detectors, two sets of features

Run the next cell. It runs the four detectors of this module on the raw features (amount, hour, account type), then on the eight behaviour features, with the comparison table's settings. It takes about 40 seconds.

> **Predict.** Which gives the larger gain: changing the detector, or changing the features?

> **Check.** You should see Isolation Forest at 0.208 with the raw features and 0.500 with the behaviour features.

In [ ]:
from sklearn.preprocessing import StandardScaler
from pyod.models.knn import KNN
from pyod.models.lof import LOF
from pyod.models.iforest import IForest
from pyod.models.ocsvm import OCSVM
raw = features.raw(payments)
raw["amount"] = np.log(raw.amount)

def run_four(X, label):
    """The four detectors of this module with the comparison table's settings; returns the precision of each."""
    scaler = StandardScaler().fit(X.loc[reference.index])
    X_ref, X_ev = scaler.transform(X.loc[reference.index]), scaler.transform(X.loc[evaluation.index])
    sample = np.random.default_rng(0).choice(len(X_ref), 20_000, replace=False)
    detectors = {"kNN distance": (KNN(n_neighbors=10, method="largest"), X_ref[sample]),
                 "LOF": (LOF(n_neighbors=35, novelty=True), X_ref[sample]),
                 "Isolation Forest": (IForest(n_estimators=200, random_state=0, n_jobs=-1), X_ref),
                 "One-Class SVM": (OCSVM(kernel="rbf", nu=0.02, gamma="auto"), X_ref[sample[:10_000]])}
    out = {}
    for name, (model, train) in detectors.items():
        row = compare.score_payments(evaluation, model.fit(train).decision_function(X_ev), name, label, explain_minutes=3)
        table.add(row)
        out[name] = row["precision at budget"]
    return out

table = compare.Table()
results = pd.DataFrame({"raw": run_four(raw, "raw features"), "behaviour": run_four(behaviour, "behaviour features")})
table.show()
results["gain"] = results.behaviour - results.raw
print(results.round(3))

With the raw features, the best and the worst detectors are 0.051 apart. With the behaviour features, the three good detectors are only 0.025 apart. But the features move each good detector by about 0.29. The features decided more than the algorithm. LOF is the exception: it gets **worse** with better features, for the reason of the LOF lesson.

## 3. Check yourself

**Your turn 1.** What is the gain of Isolation Forest from raw to behaviour features? Write it as `gain_forest` (3 decimals).

In [ ]:
gain_forest = ...   # your answer

In [ ]:
expect('gain_forest', gain_forest, 0.292, tolerance=0.001)

**Your turn 2.** Which single behaviour feature helps most? Add each behaviour feature (except `is_business`, which the raw features already have) to the raw features, one at a time, and run Isolation Forest. Put the precision of each in a dictionary `plus_one` (feature → precision), and the best feature in `most_useful`. This cell takes about 30 seconds.

In [ ]:
plus_one = ...    # your code
most_useful = ...

In [ ]:
expect_hash('most_useful', most_useful, '8b8edc41bbcba874')

In the solution: new_country 0.437, new_device 0.406, device_payments_last_hour 0.404, payments_last_hour 0.308, log_minutes_since_last 0.281, log_amount_vs_usual 0.256, night 0.199. One well-chosen feature (new_country) moves Isolation Forest from 0.208 to 0.437: more than any change of algorithm in this module.

**Your turn 3.** Write two sentences for Sofia: which gave the larger gain on Larkfield's payments, the better features or the better algorithm, with two numbers from the table above. (No check: compare your answer with the lesson.)

## Recap

- Behaviour features compare a payment with the customer's own past: the usual amount, a new country or device, the time since the last payment, the payments in the last hour, the night.
- With the raw features, every detector is near 0.2. With the behaviour features, the good ones are near 0.5, against a ceiling of 0.542.
- The features gave about +0.29 to each good detector; choosing among the good detectors changed 0.025.

The next module moves from single payments to series in time: daily orders, tickets and metrics, where "normal" changes with the hour and the season.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Normal changes with the hour and the season](https://learning.nextia-ai.com/courses/anomaly-detection/m05/normal-changes-with-the-hour-and-the-season/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/anomaly-detection/m04/features-decide-more-than-the-algorithm/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.